# 시퀀스가 길어질 때 — KV 캐시 대 고정 상태

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/mamba3_inference_cost.ipynb)

Mamba-3: Improved Sequence Modeling using State Space Principles
([arXiv 2603.15569](https://arxiv.org/abs/2603.15569), ICLR 2026)
을 읽다 보면 나오는 비용 구조를, 최소 구현 두 개로 직접 재 봅니다.

확인하려는 것은 하나입니다 — **이미 본 토큰이 L개일 때 다음 토큰
하나를 뽑는 비용이 L 에 따라 어떻게 변하는가.**

| 방식 | 들고 있어야 하는 것 | L 에 대해 |
|---|---|---|
| 어텐션 | 지금까지의 K·V 전부 | 선형으로 증가 |
| 선형 순환 | 상태 텐서 하나 | 변하지 않음 |

**이 노트북은 논문의 성능을 재현하지 않습니다.** 정확도·perplexity
는 재지 않고, 학습된 가중치도 쓰지 않습니다. 무작위 초기화한
최소 구현으로 **비용 구조**만 봅니다. 여기 나오는 숫자를 Mamba-3
의 성능으로 읽으면 안 됩니다.

`런타임 → 런타임 유형 변경 → T4 GPU` 를 켜고 실행하세요.

In [ ]:
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print("torch  :", torch.__version__)
print("numpy  :", np.__version__)

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device :", dev)
if dev == "cuda":
    p = torch.cuda.get_device_properties(0)
    print("gpu    :", p.name)
    print("memory :", p.total_memory // 1024 ** 2, "MiB")
    print("sdpa flash    :",
          torch.backends.cuda.flash_sdp_enabled())
    print("sdpa mem_eff  :",
          torch.backends.cuda.mem_efficient_sdp_enabled())
    print("sdpa math     :",
          torch.backends.cuda.math_sdp_enabled())

## 설정

두 방식의 은닉 차원을 같게 맞춥니다. 층 하나 기준입니다 —
실제 모델은 이 값에 층 수를 곱하면 됩니다.

In [ ]:
D_MODEL = 1024
N_HEADS = 16
D_HEAD = D_MODEL // N_HEADS
D_STATE = 128
# GPU 는 fp16, CPU 는 fp16 커널이 없어 fp32 를 쓴다
DTYPE = torch.float16 if dev == "cuda" else torch.float32

LENS = [1024, 2048, 4096, 8192,
        16384, 32768, 65536]

print("d_model :", D_MODEL)
print("n_heads :", N_HEADS, "· d_head :", D_HEAD)
print("d_state :", D_STATE)
print("dtype   :", DTYPE)
print("lengths :", LENS)
print("배수    :", LENS[-1] // LENS[0], "배")

## 한 스텝 디코딩 — 두 방식을 나란히

두 클래스 모두 "토큰 1개를 받아 1개를 내놓는" 같은 인터페이스
입니다. 다른 건 `alloc` 이 무엇을 잡느냐, `forward` 가 그걸
어떻게 쓰느냐입니다. 정의한 뒤 한 번씩 돌려 모양을 확인합니다.

In [ ]:
class AttnStep(nn.Module):
    """길이 L 의 KV 캐시를 들고 1토큰을 디코딩한다."""

    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(D_MODEL, 3 * D_MODEL,
                             bias=False)
        self.out = nn.Linear(D_MODEL, D_MODEL, bias=False)

    def alloc(self, L):
        shape = (1, N_HEADS, L, D_HEAD)
        k = torch.zeros(shape, device=dev, dtype=DTYPE)
        v = torch.zeros(shape, device=dev, dtype=DTYPE)
        return [k, v, L - 1]        # L 에 비례해 커진다

    def forward(self, x, state):
        k_c, v_c, pos = state
        q, k, v = self.qkv(x).chunk(3, -1)
        s = (1, 1, N_HEADS, D_HEAD)
        q = q.view(s).transpose(1, 2)
        k_c[:, :, pos] = k.view(s).transpose(1, 2)[:, :, 0]
        v_c[:, :, pos] = v.view(s).transpose(1, 2)[:, :, 0]
        o = F.scaled_dot_product_attention(
            q, k_c[:, :, :pos + 1], v_c[:, :, :pos + 1])
        o = o.transpose(1, 2).reshape(1, 1, D_MODEL)
        return self.out(o)


class RecurStep(nn.Module):
    """고정 크기 상태만 들고 1토큰을 디코딩한다."""

    def __init__(self):
        super().__init__()
        self.inp = nn.Linear(D_MODEL, 2 * D_MODEL,
                             bias=False)
        self.bc = nn.Linear(D_MODEL, 2 * D_STATE,
                            bias=False)
        self.dtp = nn.Linear(D_MODEL, N_HEADS, bias=False)
        self.out = nn.Linear(D_MODEL, D_MODEL, bias=False)
        a = torch.arange(1.0, D_STATE + 1.0)
        self.register_buffer("A", -a)

    def alloc(self, L):
        shape = (1, N_HEADS, D_HEAD, D_STATE)
        h = torch.zeros(shape, device=dev, dtype=DTYPE)
        return [h, L - 1]           # L 과 무관하게 같다

    def forward(self, x, state):
        h, _ = state
        u, g = self.inp(x).chunk(2, -1)
        b, cproj = self.bc(x).chunk(2, -1)   # SSM 의 B·C
        dt = F.softplus(self.dtp(x).float())
        dt = dt.view(1, N_HEADS, 1, 1)
        dA = torch.exp(dt * self.A).to(DTYPE)
        u = u.view(1, N_HEADS, D_HEAD, 1)
        b = b.view(1, 1, 1, D_STATE)
        h.mul_(dA)                  # 상태를 제자리에서 갱신
        h.add_(dt.to(DTYPE) * u * b)
        cproj = cproj.view(1, 1, 1, D_STATE)
        y = (h * cproj).sum(-1).reshape(1, 1, D_MODEL)
        return self.out(y * F.silu(g))

torch.manual_seed(0)
attn = AttnStep().to(dev).to(DTYPE).eval()
recur = RecurStep().to(dev).to(DTYPE).eval()
recur.A = recur.A.float()   # 감쇠 항만 fp32 로 둔다

x = torch.randn(1, 1, D_MODEL, device=dev, dtype=DTYPE)

with torch.inference_mode():
    ya = attn(x, attn.alloc(1024))
    yr = recur(x, recur.alloc(1024))

for tag, mod, y in (("attn", attn, ya),
                    ("recur", recur, yr)):
    n = sum(p.numel() for p in mod.parameters())
    print(f"{tag:<6} out {tuple(y.shape)}"
          f"  params {n:,}")

## 들고 있어야 하는 크기

계산식이 아니라 실제로 잡은 텐서의 바이트를 잽니다.

- 어텐션 — `2 · L · n_heads · d_head · 2바이트` (K 와 V)
- 선형 순환 — `n_heads · d_head · d_state · 2바이트` (L 무관)

In [ ]:
def mib(cache):
    return sum(t.numel() * t.element_size()
               for t in cache
               if torch.is_tensor(t)) / 1024 ** 2


for L in (LENS[0], LENS[-1]):
    a = mib(attn.alloc(L))
    r = mib(recur.alloc(L))
    print(f"L={L:>6}  attn {a:8.2f} MiB"
          f"  recur {r:6.2f} MiB")

## 측정

길이마다 워밍업 10회를 버리고 50회를 재 중앙값을 씁니다.
GPU 는 비동기라 매번 `synchronize` 로 끝을 기다립니다.

In [ ]:
WARM = 10
REPEAT = 50


def sync():
    if dev == "cuda":
        torch.cuda.synchronize()


def bench(mod, L):
    c = mod.alloc(L)
    with torch.inference_mode():
        for _ in range(WARM):
            mod(x, c)
        sync()
        ts = []
        for _ in range(REPEAT):
            t0 = time.perf_counter()
            mod(x, c)
            sync()
            ts.append((time.perf_counter() - t0) * 1e3)
    size = mib(c)
    del c
    if dev == "cuda":
        torch.cuda.empty_cache()
    return float(np.median(ts)), size


rows = []
for L in LENS:
    at, am = bench(attn, L)
    rt, rm = bench(recur, L)
    rows.append((L, at, rt, am, rm))
    print(f"L={L:>6}  attn {at:6.3f} ms / {am:8.2f} MiB"
          f"   recur {rt:6.3f} ms / {rm:5.2f} MiB")

## 결과 표

In [ ]:
h1 = f"{'length':>7}{'attn (ms)':>12}{'recur (ms)':>12}"
print(h1)
print("-" * len(h1))
for L, at, rt, _, _ in rows:
    print(f"{L:>7}{at:>12.3f}{rt:>12.3f}")

print()
h2 = f"{'length':>7}{'attn (MiB)':>12}{'recur (MiB)':>13}"
print(h2)
print("-" * len(h2))
for L, _, _, am, rm in rows:
    print(f"{L:>7}{am:>12.2f}{rm:>13.2f}")

print()
f_t = rows[-1][1] / rows[0][1]
f_m = rows[-1][3] / rows[0][3]
print(f"어텐션  {LENS[0]} -> {LENS[-1]}"
      f" : 시간 {f_t:.1f}배 · 캐시 {f_m:.1f}배")
f_t = rows[-1][2] / rows[0][2]
f_m = rows[-1][4] / rows[0][4]
print(f"선형순환 {LENS[0]} -> {LENS[-1]}"
      f" : 시간 {f_t:.1f}배 · 상태 {f_m:.1f}배")

## 그래프

In [ ]:
import matplotlib.pyplot as plt

os.makedirs("out", exist_ok=True)
L_ax = [r[0] for r in rows]

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(L_ax, [r[1] for r in rows], "o-",
        color="tab:red", label="attention (KV cache)")
ax.plot(L_ax, [r[2] for r in rows], "s-",
        color="tab:blue", label="linear recurrence")
ax.set_xscale("log", base=2)
ax.set_xlabel("sequence length (tokens)")
ax.set_ylabel("decode 1 token (ms)")
ax.set_ylim(bottom=0)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig("out/mamba3_decode_time.png", dpi=150)
print("saved: out/mamba3_decode_time.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(L_ax, [r[3] for r in rows], "o-",
        color="tab:red", label="attention (KV cache)")
ax.plot(L_ax, [r[4] for r in rows], "s-",
        color="tab:blue", label="linear recurrence")
ax.set_xscale("log", base=2)
ax.set_yscale("log", base=2)
ax.set_xlabel("sequence length (tokens)")
ax.set_ylabel("state to keep (MiB)")
ax.grid(alpha=0.25, which="both")
ax.legend()
fig.tight_layout()
fig.savefig("out/mamba3_state_size.png", dpi=150)
print("saved: out/mamba3_state_size.png")
plt.show()

## 읽는 법

- 크기 쪽은 구현에 좌우되지 않습니다. 어텐션은 K·V 를 다 들고
  있어야 하니 L 에 정비례하고, 선형 순환은 상태 하나라 평평
  합니다. 위 표의 배수를 그대로 보면 됩니다.
- 시간 쪽은 커널과 하드웨어를 탑니다. 어텐션은
  `F.scaled_dot_product_attention` 으로 재는데, PyTorch 가
  백엔드를 자동으로 고릅니다(맨 위 셀에 어떤 백엔드가 켜져
  있는지 찍힙니다). 짧은 길이에서는 두 방식 모두 커널 실행
  비용에 묻혀 차이가 잘 안 보입니다.
- 층 하나 기준입니다. 층이 24개면 크기도 24배입니다.
- 다시 강조하면 이 숫자는 **비용 구조**이지 모델 품질이
  아닙니다. 무작위 가중치이고 정확도는 재지 않았습니다.